# Example run of DT-LEAMBAS GPU

Example of running the LEMBAS model on toy macrophage data.

In [1]:
import os

import numpy as np
import pandas as pd

from scipy.stats import pearsonr
import torch
import torch.nn as nn

import plotnine as p9

In [2]:
!pwd

/nobackup/users/meimetis/LiverLEMBAS/learning


In [3]:
import sys 
gpu_lembas_path = '/nobackup/users/meimetis/LEMBAS-main'
sys.path.insert(1,os.path.join(gpu_lembas_path))
from LEMBAS.model.bionetwork import format_network, SignalingModel
from LEMBAS.model.train import train_signaling_model
import LEMBAS.utilities as utils
from LEMBAS import plotting, io

In [4]:
import time
import argparse
import logging
from pathlib import Path

In [5]:
args = {
    'DrugsIn':'../preprocessing/preprocessed_data/TrainingValidationData/Hepatocytes_input_conditions.tsv',
    'TargetsIn':'../preprocessing/preprocessed_data/PKN/Hepatocytes_DTI_mask.tsv',
    'TFsOut':'../preprocessing/preprocessed_data/TF_activities/Trimmed_TFacitivities_hepatocytes_microarray.tsv',
    'ChemicalSims':'../preprocessing/preprocessed_data/ChemicalSims/microarray_similarities.csv',
    'PknAnnotation':'../preprocessing/preprocessed_data/PKN/Hepatocytes-PKN-Annotation.tsv',
    'PKN':'../preprocessing/preprocessed_data/PKN/Hepatocytes-PKN-Model.tsv',
    'infoPath':'../preprocessing/preprocessed_data/TrainingValidationData/Hepatocytes_inputdataInfo.tsv',
    'res_dir':'../results/',
    'outPattern':'hepatocytes_dtlembas',
    'no':0,
    'batchSize':35,
    'MoAFactor':0.1,
    'spectralFactor':1e-3,
    'maxIter':1000,
    'noiseLevel':10,
    'L2':1e-6,
    'lamda':1e-7,
    'ConvertToEmpProb':False,
    'inputAmplitude':3,
    'projectionAmplitude':1.2,
    'max_LEMBAS_iterations':100,
    'LEMBAS_leak':0.01,
    'LEMBAS_tolerance':1e-5,
    'drug_bn' : 0.2,
    'drug_drop': 0.1,
    'timepoint':'8 hour',
    'test_dose':'Middle',    
}

In [6]:
DrugsIn = args['DrugsIn']
TargetsIn = args['TargetsIn']
TFsOut = args['TFsOut']
ChemicalSims = args['ChemicalSims']
PknAnnotation = args['PknAnnotation']
PKN = args['PKN']
res_dir = args['res_dir']
infoPath = args['infoPath']
outPattern = args['outPattern']
# outPattern = outPattern + str(args['no'])

In [7]:
logging.basicConfig(level=logging.INFO, format='%(message)s')
logger = logging.getLogger()
print2log = logger.info

# parser = argparse.ArgumentParser(prog='Cell-line simulation')
# parser.add_argument('--DrugsIn', action='store',type = str, default='../preprocessing/preprocessed_data/TrainingValidationData/Hepatocytes_input_conditions.tsv')
# parser.add_argument('--TargetsIn', action='store',type = str, default='../preprocessing/preprocessed_data/PKN/Hepatocytes_DTI_mask.tsv')
# parser.add_argument('--TFsOut', action='store',type = str, default='../preprocessing/preprocessed_data/TF_activities/Trimmed_TFacitivities_hepatocytes_microarray.tsv')
# parser.add_argument('--ChemicalSims', action='store',type = str, default='../preprocessing/preprocessed_data/ChemicalSims/microarray_similarities.csv')
# parser.add_argument('--PKN', action='store',type = str, default='../preprocessing/preprocessed_data/PKN/Hepatocytes-PKN-Model.tsv')
# parser.add_argument('--PknAnnotation', action='store',type = str, default='../preprocessing/preprocessed_data/PKN/Hepatocytes-PKN-Annotation.tsv')
# parser.add_argument('--infoPath', action='store',type = str, default='../preprocessing/preprocessed_data/TrainingValidationData/Hepatocytes_inputdataInfo.tsv')
# parser.add_argument('--res_dir', action='store',type = str, default='../results/')
# parser.add_argument('--model_type', action='store',type = int, default=4)
# parser.add_argument('--outPattern', action='store',type = str, default='hepatocytes_dtlembas')
# parser.add_argument('--no', action='store',type = int, default=0)
# parser.add_argument('--batchSize', action='store',type = int, default=25)
# parser.add_argument('--MoAFactor', action='store',type = float, default=0.1)
# parser.add_argument('--spectralFactor', action='store', default=1e-3)
# parser.add_argument('--maxIter', action='store',type = int, default=2000)
# parser.add_argument('--noiseLevel', action='store',type = float, default=10)
# parser.add_argument('--L2', action='store', default=1e-6)
# parser.add_argument('--lamda', action='store',type = float, default=0.005)
# parser.add_argument('--ConvertToEmpProb', action='store',type = bool, default=False)
# parser.add_argument('--inputAmplitude', action='store',type = float, default=3)
# parser.add_argument('--projectionAmplitude', action='store',type = float, default=1.2)
# parser.add_argument('--LEMBAS_iterations', action='store',type = int, default=120)
# parser.add_argument('--LEMBAS_leak', action='store',type = float, default=0.01)
# parser.add_argument('--LEMBAS_targetPrecision', action='store', default=1e-6)
# parser.add_argument('--drug_bn', action='store', default=0.6)
# parser.add_argument('--drug_drop', action='store', default=0.1)
# parser.add_argument('--timepoint', action='store',type=str, default='8 hour')
# parser.add_argument('--test_dose', action='store',type=str, default='Middle')
# args = parser.parse_args()
# DrugsIn = args.DrugsIn
# TargetsIn = args.TargetsIn
# TFsOut = args.TFsOut
# ChemicalSims = args.ChemicalSims
# PknAnnotation = args.PknAnnotation
# PKN = args.PKN
# res_dir = args.res_dir
# infoPath = args.infoPath
# outPattern = args.outPattern
# outPattern = outPattern + str(args.no)

In [8]:
n_cores = 12
utils.set_cores(n_cores)

seed = 888
if seed:
    torch.use_deterministic_algorithms(True)
    os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
    utils.set_seeds(seed = seed)

device = "cuda" if torch.cuda.is_available() else "cpu"
data_path = res_dir
if not os.path.isdir(data_path):
    os.mkdir(data_path)

In [9]:
# timepoint = args.timepoint
# test_dose = args.test_dose
# res_dir = res_dir + test_dose
timepoint = args['timepoint']
test_dose = args['test_dose']

In [10]:
projection_amplitude_in = args['inputAmplitude']
projection_amplitude_out = args['projectionAmplitude']
ConvertToEmpProb = args['ConvertToEmpProb']

In [11]:
#Setup optimizer
batchSize = args['batchSize']
MoAFactor = args['MoAFactor']
spectralFactor = args['spectralFactor']
maxIter = args['maxIter']
L2 = args['L2']
lamda = args['lamda']

In [12]:
Path(res_dir).mkdir(parents=True, exist_ok=True)

In [13]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [14]:
def getSamples(N, batchSize):
    order = np.random.permutation(N)
    outList = []
    while len(order)>0:
        outList.append(order[0:batchSize])
        order = order[batchSize:]
    return outList

In [15]:
def pearson_r(y_true, y_pred):
    x = y_true
    y = y_pred
    mx = torch.mean(x, dim=0)
    my = torch.mean(y, dim=0)
    xm, ym = x - mx, y - my
    r_num = torch.sum(xm * ym,dim=0)
    x_square_sum = torch.sum(xm * xm,dim=0)
    y_square_sum = torch.sum(ym * ym,dim=0)
    r_den = torch.sqrt(x_square_sum * y_square_sum)
    r = r_num / r_den
    return r #torch.mean(r)

def numpy_pearson_r(y_true, y_pred):
    x = y_true
    y = y_pred
    # Compute means for each column
    mx = np.mean(x, axis=0)
    my = np.mean(y, axis=0)
    # Subtract means (centering)
    xm = x - mx
    ym = y - my
    # Numerator: sum of element-wise products of centered vectors
    r_num = np.sum(xm * ym, axis=0)
    # Denominator: square root of the product of sum of squares of centered vectors
    r_den = np.sqrt(np.sum(xm * xm, axis=0) * np.sum(ym * ym, axis=0))
    # Pearson correlation for each column
    r = r_num / r_den
    return r

# Define drugLayer

In [16]:
class DrugstoSignal(nn.Module):
    def __init__(self,mask, size1,size2,drugSim,momentum=0.6,drop=0.1,
                 device = 'cuda' if torch.cuda.is_available() else 'cpu'):

        super().__init__()

        self.size_in = size1 ### drugs number
        self.size_out = size2 ### targets/signal size
        self.mask = mask.double().to(device)
        self.drugSim = drugSim.to(device)
        self.leak = 0.01

        # initialize weights and biases
        weights = torch.Tensor(size2,size1).double()
        nn.init.uniform_(weights, -0.3, 0.01) # weight init
        weights = torch.mul(weights.to(device),self.mask)
        ecfp4_weights = torch.Tensor(size1,size1).double()
        nn.init.uniform_(ecfp4_weights, 0., 1.)
        self.Wsim = nn.Parameter(ecfp4_weights, requires_grad=True)
        self.bn = nn.BatchNorm1d(num_features=size1, momentum=momentum, dtype=torch.double)
        self.dropout = nn.Dropout(drop)

        self.A = nn.Parameter(weights, requires_grad = True)

    def forward(self, x):
        W = torch.mul(self.drugSim, self.Wsim)
        A = torch.mul(self.A,self.mask)
        xhat = torch.mm(x,W)
        xhat = self.bn(xhat)
        xhat = self.dropout(xhat)
        xhat=xhat.transpose(0, 1)
        xhat = torch.mm(A, xhat)
        output = xhat.transpose(0, 1)
        return output

## Build the Model

### Load Input Data

In [17]:
# prior knowledge signaling network
net = pd.read_csv(PKN, sep = '\t', index_col = False).loc[:,['source','target','stimulation','inhibition']]

# ligand input and TF output
drugInput = pd.read_csv(DrugsIn, sep = '\t', index_col = 0)
drugSmiles = drugInput.columns.values
TFOutput = pd.read_csv(TFsOut, sep='\t', low_memory=False, index_col=0)
TFOutput = TFOutput.loc[drugInput.index,:]

# Load drugs information
drugTargets = pd.read_csv(TargetsIn, sep='\t', low_memory=False, index_col=0)
inputInfo = pd.read_csv(infoPath, sep='\t', low_memory=False, index_col=0)
inputInfo = inputInfo[inputInfo['time']==timepoint]
TFOutput = TFOutput.loc[inputInfo.index,:]
drugInput = drugInput.loc[inputInfo.index,:]

Let's see what the signaling network looks like:

In [18]:
stimulation_label = 'stimulation'
inhibition_label = 'inhibition'
weight_label = 'mode_of_action'
source_label = 'source'
target_label = 'target'

net[[source_label, target_label, stimulation_label, inhibition_label]].head()

,source,target,stimulation,inhibition
0,Q13255,P48995,1,0
1,P49137,Q16539,0,0
2,Q16539,P49137,1,0
3,P49137,O95453,0,1
4,O60674,P19235,1,0


Let's format it to fit with the necessary inputs to the SignalingModel:

In [19]:
net = format_network(net, weight_label = weight_label, stimulation_label = stimulation_label, inhibition_label = inhibition_label)
net[[source_label, target_label, weight_label, stimulation_label, inhibition_label]].head()

,source,target,mode_of_action,stimulation,inhibition
0,Q13255,P48995,1.0,1,0
1,P49137,Q16539,0.1,0,0
2,Q16539,P49137,1.0,1,0
3,P49137,O95453,-1.0,0,1
4,O60674,P19235,1.0,1,0


In [20]:
nodeNames = np.union1d(net.source.values,net.target.values)

In [21]:
#Subset input and output to intersecting nodes
druginName = drugInput.columns.values
inName = drugTargets.columns.values
outconds = drugInput.index.values
outName = TFOutput.columns.values
inName = np.intersect1d(nodeNames, inName)
outName = np.intersect1d(nodeNames, outName)
TFOutput = TFOutput.loc[outconds,outName]
drugTargets = drugTargets.loc[:,inName]
if ConvertToEmpProb==True:
    print2log('Convereted to Emprirical probabilities')
    TFOutput = 1/(1+np.exp(-TFOutput))

drugTargets = drugTargets.loc[drugInput.columns.values,:]
drugSim = pd.read_csv(ChemicalSims,index_col=0)
drugSim = drugSim.loc[drugInput.columns.values,drugInput.columns.values]
drugSim = torch.tensor(drugSim.values.copy(), dtype=torch.double)

In [22]:
### Split train and test set
print2log('Split train and test set...')
print2log('Kept ' + test_dose + ' as a test set!')
train_ind= inputInfo[~inputInfo.index.str.contains(test_dose)].index.tolist()
test_ind = np.setdiff1d(inputInfo.index.values, train_ind)
drugInput_train = drugInput.loc[train_ind,:]
drugInput_test = drugInput.loc[test_ind,:]
TFOutput_train = TFOutput.loc[train_ind,:]
TFOutput_test = TFOutput.loc[test_ind,:]
X = torch.tensor(drugInput.values.copy(), dtype=torch.double).to(device)
Y = torch.tensor(TFOutput.values, dtype=torch.double).to(device)
# X = torch.tensor(drugInput_train.values.copy(), dtype=torch.double).to(device)
# Y = torch.tensor(TFOutput_train.values, dtype=torch.double).to(device)
X_test = torch.tensor(drugInput_test.values.copy(), dtype=torch.double).to(device)
Y_test = torch.tensor(TFOutput_test.values, dtype=torch.double).to(device)

#### Random shuffle X
#X = X[torch.randperm(X.size()[0]),:]

Split train and test set...
Kept Middle as a test set!


In [23]:
mask_dt =torch.tensor(np.array(drugTargets)).transpose(0,1)

### Define Parameters to Build and Train Model

In [24]:
# other parameters
bionet_params = {'target_steps': 100, 'max_steps': args['max_LEMBAS_iterations'], 
                 'exp_factor':50, 'tolerance': args['LEMBAS_tolerance'], 
                 'leak':args['LEMBAS_leak']} # fed directly to model

# training parameters
lr_params = {'max_iter': args['maxIter'], 
             'learning_rate': 5e-3}
other_params = {'batch_size': args['batchSize'], 'noise_level': args['noiseLevel'], 'gradient_noise_level': 1e-9}
regularization_params = {'param_lambda_L2': args['L2'], 'moa_lambda_L1': args['MoAFactor'], 
                         'ligand_lambda_L2': 1e-5, 'uniform_lambda_L2': 1e-5, 
                   'uniform_max': 1/projection_amplitude_out, 'spectral_loss_factor': args['spectralFactor']}
spectral_radius_params = {'n_probes_spectral': 5, 'power_steps_spectral': 50, 'subset_n_spectral': 10}
target_spectral_radius = 0.8
hyper_params = {**lr_params, **other_params, **regularization_params, **spectral_radius_params} # fed into training function

### Initialize Model
Next, let's initialize the model and format the inputs/outputs for running the model:

In [25]:
mod = SignalingModel(net = net,
                     X_in = drugTargets,
                     y_out = TFOutput, 
                     projection_amplitude_in = projection_amplitude_in, projection_amplitude_out = projection_amplitude_out,
                     weight_label = weight_label, source_label = source_label, target_label = target_label,
                     bionet_params = bionet_params, 
                     dtype = torch.double, device = device, seed = seed)

The ligand input, after filtering for nodes in the network, looks like this:

In [26]:
mod.X_in.head()

,O14920,O15111,O75469,P00533,P00918,P02766,P02788,P03372,P04035,P04637,...,Q13873,Q71U36,Q92731,Q92769,Q96P20,Q99527,Q9BVA1,Q9HC97,Q9UPY5,Q9Y478
Brc1ccccc1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
C#C[C@]1(O)CC[C@H]2[C@@H]3CCC4=Cc5oncc5C[C@]4(C)[C@H]3CC[C@@]21C,0,0,0,1,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
C#C[C@]1(O)CC[C@H]2[C@@H]3CCc4cc(O)ccc4[C@H]3CC[C@@]21C,0,0,1,1,0,0,0,1,1,0,...,0,0,1,0,0,0,0,0,0,0
C[C@](N)(Cc1ccc(O)c(O)c1)C(=O)O,0,0,0,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
C[C@]12CCC(=O)C=C1CC[C@@H]1[C@@H]2CC[C@@]2(C)[C@H]1CC[C@]2(C)O,0,0,0,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0


The TF activity output, after filtering for nodes in the network, looks like this:

In [27]:
mod.y_out.head()

,O00327,O00716,O14593,O43524,O75030,P01100,P01106,P03372,P04150,P04637,...,Q15672,Q16236,Q16665,Q92731,Q99626,Q99814,Q9H3D4,Q9NQB0,Q9UJU2,Q9Y261
phenylbutazone+Control+8 hour,0.636990,0.605001,0.052481,0.528969,0.654028,0.081254,0.434636,0.518754,0.432429,0.382191,...,0.410138,0.134263,0.718960,0.768253,0.361637,0.673840,0.266724,0.705519,0.216093,0.885699
phenylbutazone+Low+8 hour,0.748591,0.545047,0.060999,0.477952,0.548814,0.038788,0.364966,0.644417,0.319034,0.322228,...,0.425977,0.136692,0.778963,0.727179,0.214288,0.600723,0.143003,0.622142,0.141544,0.875796
phenylbutazone+Middle+8 hour,0.687968,0.574259,0.120751,0.525163,0.586153,0.064608,0.276370,0.614758,0.493113,0.337953,...,0.513500,0.113049,0.794481,0.793970,0.315999,0.839430,0.170239,0.576894,0.166308,0.861977
phenylbutazone+High+8 hour,0.883097,0.430459,0.082136,0.863760,0.680425,0.062188,0.177927,0.558325,0.530832,0.552992,...,0.807415,0.288992,0.973237,0.815037,0.286405,0.896593,0.363280,0.447375,0.052014,0.890756
simvastatin+Control+8 hour,0.790973,0.211745,0.312000,0.877484,0.316155,0.630080,0.964468,0.147742,0.178061,0.425165,...,0.577062,0.846870,0.474776,0.455408,0.661795,0.415356,0.342057,0.363922,0.299543,0.105410


A forward pass looks like this:

In [28]:
# X_in = mod.df_to_tensor(mod.X_in) # ligand inputs
# X_full = mod.input_layer(X_in) # ligand inputs in signaling network
# Y_full = mod.signaling_network(X_full) # signaling network weights
# Y_hat = mod.output_layer(Y_full) # TF outputs of signaling network

In [29]:
drugLayer = DrugstoSignal(mask_dt, 
                          mask_dt.shape[1],mask_dt.shape[0],
                          drugSim,momentum=args['drug_bn'],drop=args['drug_drop'],
                          device = device)
drugLayer = drugLayer.to(device)

In [30]:
# Xin = drugLayer(X)

In [31]:
# Yhat, Yhatfull =mod(Xin)

### Train Model

In [32]:
stats = utils.initialize_progress(hyper_params['max_iter'])
# model setup
mod.input_layer.weights.requires_grad = False # don't learn scaling factors for the ligand input concentrations
mod.signaling_network.prescale_weights(target_radius = target_spectral_radius) # spectral radius

# loss and optimizer
criterion = torch.nn.MSELoss(reduction='mean')
optimizer = torch.optim.Adam(list(mod.parameters())+list(drugLayer.parameters()), 
                             lr=1,weight_decay=0)
resetState = optimizer.state.copy()

In [33]:
mLoss = criterion(torch.mean(Y, dim=0) * torch.ones(Y.shape, device = Y.device), Y)
N = X.shape[0]

In [ ]:
print2log('Begin training...')
print2log(mLoss.item())
start_time = time.time()
reset_epoch = 200
reset_state = optimizer.state.copy()
for e in range(hyper_params['max_iter']):
    # set learning rate
    cur_lr = utils.get_lr(e, hyper_params['max_iter'], max_height = hyper_params['learning_rate'],
                          start_height=hyper_params['learning_rate']/10, end_height=1e-4, peak = 500)
    optimizer.param_groups[0]['lr'] = cur_lr
        
    cur_loss = []
    cur_eig = []
    
    trainloader = getSamples(N, batchSize)
    for dataIndex in trainloader:
        mod.train()
        drugLayer.train()
        #mod.signaling_network.weights = mod.signaling_network.weights + 1e-8 * torch.randn(mod.signaling_network.weights.shape) #breaks potential symmetries
        optimizer.zero_grad()

        dataIn = X[dataIndex, :].view(len(dataIndex), X.shape[1])
        dataOut = Y[dataIndex, :].view(len(dataIndex), Y.shape[1])
        
        Xin = drugLayer(dataIn)
        X_full = mod.input_layer(Xin)
        network_noise = torch.randn(X_full.shape, device = X_full.device)
        X_full = X_full + hyper_params['noise_level'] * cur_lr * network_noise
        YhatFull = mod.signaling_network(X_full)
        Yhat = mod.output_layer(YhatFull)
        
        # get prediction loss
        fit_loss = criterion(dataOut, Yhat)
            
        # get regularization losses
        sign_reg = mod.signaling_network.sign_regularization(lambda_L1 = hyper_params['moa_lambda_L1']) # incorrect MoA
        ligand_reg = mod.ligand_regularization(lambda_L2 = hyper_params['ligand_lambda_L2']) # ligand biases
        stability_loss, spectral_radius = mod.signaling_network.get_SS_loss(Y_full = YhatFull.detach(), spectral_loss_factor = hyper_params['spectral_loss_factor'],
                                                                                subset_n = hyper_params['subset_n_spectral'], n_probes = hyper_params['n_probes_spectral'], 
                                                                                power_steps = hyper_params['power_steps_spectral'])
        uniform_reg = mod.uniform_regularization(lambda_L2 = hyper_params['uniform_lambda_L2']*cur_lr, Y_full = YhatFull, 
                                                     target_min = 0, target_max = hyper_params['uniform_max']) # uniform distribution
        param_reg = mod.L2_reg(hyper_params['param_lambda_L2']) # all model weights and signaling network biases
            
        total_loss = fit_loss + sign_reg + ligand_reg + param_reg + stability_loss + uniform_reg
    
        # gradient
        total_loss.backward()
        mod.add_gradient_noise(noise_level = hyper_params['gradient_noise_level'])
        optimizer.step()
    
        # store
        cur_eig.append(spectral_radius)
        cur_loss.append(fit_loss.item())
    
    stats = utils.update_progress(stats, iter = e, loss = cur_loss, eig = cur_eig, learning_rate = cur_lr, 
                                     n_sign_mismatches = mod.signaling_network.count_sign_mismatch())
        
    if e % 100 == 0:
        utils.print_stats(stats, iter = e)
        # store results
        io.write_pickled_object(stats, os.path.join(data_path, 'training_stats.pickle'))
        
    if np.logical_and(e % reset_epoch == 0, e>0):
        optimizer.state = reset_state.copy()

mins, secs = divmod(time.time() - start_time, 60)
print("Training ran in: {:.0f} min {:.2f} sec".format(mins, secs))
    
# store results
io.write_pickled_object(stats, os.path.join(data_path, 'training_stats.pickle'))
torch.save(obj=mod.state_dict(), f=os.path.join(data_path, 'hepatocytes_signal_state_dict.pth'))
torch.save(obj=drugLayer.state_dict(), f=os.path.join(data_path, 'drugLayer_state_dict.pth'))

Begin training...
0.0680131041583857
i=0, l=0.26924, s=0.723, r=0.00050, v=0
i=100, l=0.06664, s=0.572, r=0.00093, v=0
i=200, l=0.04128, s=0.757, r=0.00205, v=0


## Predict using the model:

In [ ]:
mod.load_state_dict(torch.load(os.path.join(data_path, 'hepatocytes_signal_state_dict.pth')))
drugLayer.load_state_dict(torch.load(os.path.join(data_path, 'drugLayer_state_dict.pth')))
stats = io.read_pickled_object(os.path.join(data_path, 'training_stats.pickle'))

In [ ]:
mod.eval()
drugLayer.eval()
# Xin  = drugLayer(X_test)
Xin  = drugLayer(X)
Y_hat, Y_full = mod(Xin)

## Visualize

Let's see how the various stats performed across training:

In [ ]:
loss_smooth = utils.get_moving_average(values = stats['loss_mean'], n_steps = 5)
loss_sigma_smooth = utils.get_moving_average(values = stats['loss_sigma'], n_steps = 10)
epochs = np.array(range(stats['loss_mean'].shape[0]))

p1A = plotting.shade_plot(X = epochs, Y = loss_smooth, sigma = loss_sigma_smooth, x_label = 'Epoch', y_label = 'Loss')
p1A += p9.scale_y_log10()
p1A += p9.geom_hline(yintercept = mLoss.item(), linetype="dashed", color="black")
p1A.draw()

As well as how the learning rate was adjusted across training:

In [ ]:
viz_df = pd.DataFrame(data = {'Epoch': epochs, 'lr': stats['learning_rate']})
width, height = 5, 3
p1B = (
    p9.ggplot(viz_df, p9.aes(x='Epoch', y = 'lr')) +
    p9.geom_line(color = '#1E90FF') +
    p9.theme_bw() + 
    p9.theme(figure_size=(width, height)) +
    p9.ylab('Learning Rate')
)
p1B.draw()

As well as the spectral radius:

In [ ]:
eig_smooth = utils.get_moving_average(stats['eig_mean'], 5)
eig_sigma_smooth = utils.get_moving_average(stats['eig_sigma'], 5)

p1C = plotting.shade_plot(X = epochs, Y = eig_smooth, sigma = eig_sigma_smooth, x_label = 'Epoch', y_label = 'Spectral Radius')
p1C += p9.geom_hline(yintercept = mod.signaling_network.training_params['spectral_target'], linetype="dashed", color="black")
p1C += p9.geom_hline(yintercept = 1, color="black")
p1C.draw()

Next, let's see what the correlation between the predicted and actual transcription factor is:

In [ ]:
y_pred = Y_hat.detach().flatten().cpu().numpy()
# y_actual = Y_test.detach().flatten().cpu().numpy()
y_actual = Y.detach().flatten().cpu().numpy()

pr, _ = pearsonr(y_pred, y_actual)

viz_df = pd.DataFrame(data = {'Predicted': y_pred, 'Actual': y_actual})
width, height = 3, 3
p2 = (
    p9.ggplot() +
    p9.geom_point(data = viz_df, mapping = p9.aes(x='Predicted', y = 'Actual'), color = '#1E90FF') +
    p9.geom_line(data = pd.DataFrame(data = {'x': [0,1], 'y': [0, 1]}), mapping = p9.aes(x='x', y = 'y'), color = 'black') +
    p9.theme_bw() + 
    p9.theme(figure_size=(width, height)) +
    p9.annotate(geom = 'text', x=0.25, y = 0.95, label = 'Pearson r: {:.2f}'.format(pr), size = 10)
)
p2.draw()

In [ ]:
r_train = pearson_r(Y, Y_hat).detach().cpu().numpy()
print2log(r_train.mean())
# r_test = pearson_r(Y_test, Yhat_test).detach().numpy()
# df_res = pd.DataFrame({'TF':TFOutput.columns,'train': r_train, 'test': r_test})
# df_res['fold'] = test_dose
# df_res['time'] = timepoint
# df_res['model_no'] = arg['no'] # args.no
# df_res.to_csv(res_dir+'/performance_'+outPattern +'.csv')

In [ ]:
width, height = 6, 6
phist = (
    p9.ggplot(data = pd.DataFrame({'r':r_train}),mapping = p9.aes(x='r')) +
    p9.geom_histogram(fill = '#1E90FF') +
    p9.scale_x_continuous(breaks=[-0.75,-0.5,-0.25,0,0.25,0.5,0.75,1])+
    p9.scale_y_continuous(breaks=[0,1,2,5,10,15,20,30,40,50,75,100])+
    p9.theme_bw() + 
    p9.theme(figure_size=(width, height)) 
)
phist.draw()

Finally, let's see how our prediction error compares to error of the prediction relative to a control predicted on an input of no ligands. 

In [ ]:
X_ctrl = torch.zeros((1, Xin.shape[1]), dtype=mod.dtype, device = mod.device)
Y_ctrl, _ = mod(X_ctrl)
signal_distance = torch.sum(torch.abs(Y_hat - Y_ctrl),dim=1).detach().cpu().numpy().flatten()
# fit_distance = torch.sum(torch.square(Y_hat-Y_test),dim=1).detach().cpu().numpy().flatten()
fit_distance = torch.sum(torch.square(Y_hat-Y),dim=1).detach().cpu().numpy().flatten()


viz_df = pd.DataFrame(data = {'Control Distance': signal_distance, 'Fit Distance': fit_distance})
width, height = 3, 3
p3 = (
    p9.ggplot() +
    p9.geom_point(data = viz_df, mapping = p9.aes(x='Control Distance', y = 'Fit Distance'), color = '#1E90FF') +
    p9.theme_bw() + 
    p9.theme(figure_size=(width, height)) 
)
p3.draw()